In [1]:
# ============================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ============================================================

!pip -q install -U transformers accelerate sentencepiece gradio

In [2]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================

import os
import gc
import re
import json
import textwrap
import traceback
from datetime import datetime, timedelta

import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# ============================================================
# CELL 3 — DEVICE DETECTION
# ============================================================

print("=" * 60)
print("AI STUDYMATE — DEVICE CHECK")
print("=" * 60)

if torch.cuda.is_available():
    device = "cuda"

    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print("✅ GPU detected")
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {total_vram:.2f} GB")

    if total_vram < 4:
        print("⚠️ GPU has limited VRAM. The model may need extra memory optimization.")

else:
    device = "cpu"

    print("⚠️ No CUDA GPU detected.")
    print("Using CPU instead.")
    print("Generation will be slower on CPU.")

print(f"\nSelected device: {device}")

AI STUDYMATE — DEVICE CHECK
✅ GPU detected
GPU: Tesla T4
VRAM: 14.56 GB

Selected device: cuda


In [4]:
# ============================================================
# CELL 4 — MODEL CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

APP_NAME = "AI StudyMate"

# Maximum input size
MAX_INPUT_TOKENS = 3072

# Default output size
DEFAULT_MAX_NEW_TOKENS = 512

# Maximum chat messages kept in memory
MAX_CHAT_MESSAGES = 12

print("Application:", APP_NAME)
print("Model:", MODEL_NAME)
print("Device:", device)

Application: AI StudyMate
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda


In [5]:
# ============================================================
# CELL 5 — DOWNLOAD AND LOAD MODEL
# ============================================================

print("=" * 60)
print("LOADING AI STUDYMATE MODEL")
print("=" * 60)

tokenizer = None
model = None
model_loaded = False
model_error = None

try:

    print("1/3 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )

    print("2/3 Loading model...")

    if device == "cuda":

        # float16 significantly reduces VRAM usage
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

    else:

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

    print("3/3 Moving model to selected device...")

    model = model.to(device)
    model.eval()

    model_loaded = True

    print("\n" + "=" * 60)
    print("✅ MODEL LOADED SUCCESSFULLY")
    print("=" * 60)

except Exception as e:

    model_error = str(e)

    print("\n❌ MODEL LOADING FAILED")
    print("-" * 60)
    print(model_error)
    print("-" * 60)

    print("""
Possible solutions:

1. Restart the Colab runtime.
2. Make sure Runtime → Change runtime type → GPU is selected.
3. Run Cell 5 again.
4. If memory is insufficient, change MODEL_NAME in Cell 4 to:

   Qwen/Qwen2.5-0.5B-Instruct

The smaller model requires substantially less memory.
""")

LOADING AI STUDYMATE MODEL
1/3 Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

3/3 Moving model to selected device...

✅ MODEL LOADED SUCCESSFULLY


In [6]:
# ============================================================
# CELL 6 — AI GENERATION ENGINE
# ============================================================

def clean_generated_text(text):
    """
    Clean common formatting artifacts from generated responses.
    """

    if not text:
        return ""

    text = text.strip()

    # Remove accidental assistant prefixes
    prefixes = [
        "Assistant:",
        "assistant:",
        "Answer:",
        "Response:"
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    return text


def generate_response(prompt, max_new_tokens=512):
    """
    Generate an answer using the locally loaded Hugging Face model.

    No API key is required.
    """

    if not model_loaded or model is None or tokenizer is None:
        return (
            "⚠️ AI model is not currently available.\n\n"
            f"Model error: {model_error or 'Unknown error'}"
        )

    if not prompt or not prompt.strip():
        return "Please enter some text or a question first."

    try:

        prompt = prompt.strip()

        # Prevent extremely large prompts
        prompt = prompt[:18000]

        messages = [
            {
                "role": "system",
                "content": (
                    "You are AI StudyMate, a friendly university study tutor. "
                    "Explain concepts accurately, clearly, and step-by-step. "
                    "Use simple language when appropriate. "
                    "Do not encourage academic cheating. "
                    "Help students understand and develop their own work."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        # Use the model's chat template
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_TOKENS
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.08,
                pad_token_id=tokenizer.eos_token_id
            )

        # Only decode newly generated tokens
        input_length = inputs["input_ids"].shape[1]

        generated_tokens = outputs[0][input_length:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

        response = clean_generated_text(response)

        if not response:
            return "I couldn't generate a response. Please try again."

        return response

    except torch.cuda.OutOfMemoryError:

        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        return (
            "⚠️ GPU memory was insufficient for this request.\n\n"
            "Try using a shorter prompt or reducing the requested output length."
        )

    except Exception as e:

        return (
            "⚠️ Generation error occurred.\n\n"
            f"Details: {str(e)}"
        )

In [7]:
# ============================================================
# CELL 7 — AI STUDY CHAT
# ============================================================

chat_history_data = []


def study_chat(message, history, subject, difficulty):

    if not message or not message.strip():
        return history, ""

    system_instruction = f"""
You are AI StudyMate's university tutor.

Subject: {subject}
Difficulty: {difficulty}

Student question:
{message}

Instructions:
- Explain step-by-step.
- Use examples when useful.
- Define difficult terminology.
- Do not assume the student already understands advanced concepts.
- If the question is about programming, show small examples when useful.
- If the question is mathematical, show the reasoning clearly.
- End with a short "Quick Check" question when appropriate.
"""

    answer = generate_response(
        system_instruction,
        max_new_tokens=600
    )

    if history is None:
        history = []

    history = list(history)

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": answer
    })

    return history, ""


def clear_chat():
    return []

In [8]:
# ============================================================
# CELL 8 — NOTES GENERATOR
# ============================================================

def generate_notes(subject, topic, difficulty, length):

    if not topic or not topic.strip():
        return "⚠️ Please enter a topic."

    prompt = f"""
Create high-quality university study notes.

Subject: {subject}
Topic: {topic}
Difficulty: {difficulty}
Desired length: {length}

Use this structure:

# {topic}

## 1. Overview
## 2. Key Concepts
## 3. Definitions
## 4. Examples
## 5. Important Points
## 6. Exam Tips
## 7. Quick Revision

Requirements:
- Explain clearly.
- Use headings and bullet points.
- Include examples where useful.
- Make the notes useful for exam revision.
"""

    return generate_response(
        prompt,
        max_new_tokens=700
    )

In [9]:
# ============================================================
# CELL 9 — SUMMARIZER
# ============================================================

def summarize_text(text):

    if not text or not text.strip():
        return "⚠️ Please paste your study material."

    # Limit extremely long material
    text = text.strip()[:20000]

    prompt = f"""
Summarize ONLY the information contained in the text below.

Do NOT introduce facts that are not present in the supplied material.

Create:

## Short Summary

## Key Points

## Important Definitions

## Important Formulas
Only include formulas explicitly present in the text.

## Exam-Focused Points

STUDY MATERIAL:
{text}
"""

    return generate_response(
        prompt,
        max_new_tokens=700
    )

In [10]:
# ============================================================
# CELL 10 — QUIZ GENERATOR
# ============================================================

def generate_quiz(subject, topic, number_questions, difficulty):

    if not topic or not topic.strip():
        return "⚠️ Please enter a topic."

    try:
        number_questions = int(number_questions)
    except:
        number_questions = 5

    number_questions = max(1, min(number_questions, 15))

    prompt = f"""
Create a university-level multiple-choice quiz.

Subject: {subject}
Topic: {topic}
Number of questions: {number_questions}
Difficulty: {difficulty}

For every question use exactly this structure:

### Question 1
Question text

A. Option
B. Option
C. Option
D. Option

**Correct Answer:** A/B/C/D

**Explanation:** Explain why the answer is correct.

Create {number_questions} questions.

Make the questions educational rather than trick questions.
"""

    return generate_response(
        prompt,
        max_new_tokens=1000
    )

In [11]:
# ============================================================
# CELL 11 — FLASHCARD GENERATOR
# ============================================================

def generate_flashcards(subject, topic, number_cards, difficulty):

    if not topic or not topic.strip():
        return "⚠️ Please enter a topic."

    try:
        number_cards = int(number_cards)
    except:
        number_cards = 10

    number_cards = max(1, min(number_cards, 20))

    prompt = f"""
Create {number_cards} university revision flashcards.

Subject: {subject}
Topic: {topic}
Difficulty: {difficulty}

Format:

### Flashcard 1
**Question:** ...
**Answer:** ...

### Flashcard 2
**Question:** ...
**Answer:** ...

Make each answer concise but useful.
Focus on important concepts, definitions, formulas, and facts.
"""

    return generate_response(
        prompt,
        max_new_tokens=900
    )

In [12]:
# ============================================================
# CELL 12 — STUDY PLANNER
# ============================================================

def generate_study_plan(
    subjects,
    exam_date,
    daily_time,
    weak_subjects,
    strong_subjects,
    study_days
):

    if not subjects or not subjects.strip():
        return "⚠️ Please enter your subjects."

    try:
        study_days = int(study_days)
    except:
        study_days = 7

    study_days = max(1, min(study_days, 60))

    prompt = f"""
Create a realistic university study plan.

Subjects:
{subjects}

Exam date:
{exam_date}

Daily study time:
{daily_time}

Weak subjects:
{weak_subjects or "Not specified"}

Strong subjects:
{strong_subjects or "Not specified"}

Number of study days:
{study_days}

Create:

# Study Plan

## Strategy

Explain how study time should be divided.

## Daily Schedule

Create a day-by-day plan.

For each day include:
- Subject
- Topic
- Study duration
- Activity
- Revision task

Give weak subjects additional attention while still maintaining strong subjects.

Include:
- Breaks
- Active recall
- Practice questions
- Revision sessions
- Final review

Make the schedule realistic rather than overloaded.
"""

    return generate_response(
        prompt,
        max_new_tokens=1200
    )

In [25]:
# ============================================================
# CELL 13 — ASSIGNMENT HELPER — FIXED
# ============================================================

def assignment_helper(question, subject, help_type):

    # Check assignment question
    if question is None or not str(question).strip():
        return "⚠️ Please enter your assignment question."

    try:
        question = str(question).strip()
        subject = str(subject or "General").strip()
        help_type = str(help_type or "Understand the question").strip()

        # Limit very long assignments
        if len(question) > 12000:
            question = question[:12000]
            warning = (
                "\n\n⚠️ Your assignment was very long, "
                "so only the first part was analyzed."
            )
        else:
            warning = ""

        prompt = (
            "You are AI StudyMate, a university study assistant.\n\n"

            "Subject: " + subject + "\n\n"

            "Assignment question:\n"
            + question +
            "\n\n"

            "Student needs help with:\n"
            + help_type +
            "\n\n"

            "Provide educational guidance using clear sections.\n\n"

            "## Understanding\n"
            "Explain what the assignment is asking.\n\n"

            "## Key Concepts\n"
            "Explain the concepts the student needs to understand.\n\n"

            "## Suggested Approach\n"
            "Give a step-by-step approach for completing the work.\n\n"

            "## Suggested Outline\n"
            "Provide an outline if appropriate.\n\n"

            "## Tips\n"
            "Give practical tips for producing a good answer.\n\n"

            "Academic integrity:\n"
            "Help the student learn and develop their own answer. "
            "Do not encourage submitting AI-generated work as their own."
        )

        result = generate_response(
            prompt,
            max_new_tokens=700
        )

        return result + warning

    except Exception as e:

        return (
            "⚠️ Assignment helper failed.\n\n"
            "Error: " + str(e)
        )


print("✅ CELL 13 SUCCESSFULLY LOADED")
print("📚 Assignment Helper is ready.")

✅ CELL 13 SUCCESSFULLY LOADED
📚 Assignment Helper is ready.


In [17]:
# ============================================================
# CELL 14 — CODE EXPLAINER — FIXED
# ============================================================

def explain_code(code, language):

    if code is None or not str(code).strip():
        return "⚠️ Please paste your code first."

    try:
        code = str(code).strip()

        if len(code) > 12000:
            code = code[:12000]
            warning = "\n\n⚠️ The code was shortened because it was very long."
        else:
            warning = ""

        prompt = (
            "You are AI StudyMate, a friendly beginner programming tutor.\n\n"
            "Programming language: " + str(language) + "\n\n"
            "Analyze this code:\n\n"
            "----- CODE START -----\n"
            + code +
            "\n----- CODE END -----\n\n"
            "Explain it using these sections:\n\n"
            "## 1. What This Code Does\n"
            "Explain the purpose of the code simply.\n\n"
            "## 2. Step-by-Step Explanation\n"
            "Explain the important parts in order.\n\n"
            "## 3. Important Sections\n"
            "Explain important variables, functions, loops, conditions, "
            "classes, or queries.\n\n"
            "## 4. Errors or Potential Problems\n"
            "Identify syntax errors, logical problems, or possible bugs.\n\n"
            "## 5. Improvements\n"
            "Suggest practical improvements.\n\n"
            "## 6. Beginner Explanation\n"
            "Explain the overall idea in very simple language.\n\n"
            "Do not unnecessarily rewrite the complete program."
        )

        result = generate_response(
            prompt,
            max_new_tokens=700
        )

        return result + warning

    except Exception as e:

        return (
            "⚠️ Code explanation failed.\n\n"
            "Error: " + str(e)
        )


print("✅ CELL 14 SUCCESSFULLY LOADED")
print("💻 Code Explainer is ready.")

✅ CELL 14 SUCCESSFULLY LOADED
💻 Code Explainer is ready.


In [19]:
# ============================================================
# CELL 15 — EXAM PREPARATION
# ============================================================

def exam_preparation(subject, topics, exam_date, difficulty):

    # Check subject
    if subject is None or not str(subject).strip():
        return "⚠️ Please enter a subject."

    # Check topics
    if topics is None or not str(topics).strip():
        return "⚠️ Please enter the topics you need to study."

    try:
        subject = str(subject).strip()
        topics = str(topics).strip()
        exam_date = str(exam_date).strip()
        difficulty = str(difficulty).strip()

        prompt = (
            "You are AI StudyMate, a university exam-preparation tutor.\n\n"
            "Create a practical exam preparation plan.\n\n"
            "Subject: " + subject + "\n"
            "Topics: " + topics + "\n"
            "Exam date: " + exam_date + "\n"
            "Difficulty: " + difficulty + "\n\n"

            "Use the following structure:\n\n"

            "# Exam Preparation Plan\n\n"

            "## 1. Priority Topics\n"
            "Rank the topics by importance and explain why.\n\n"

            "## 2. Revision Strategy\n"
            "Give a practical strategy for learning and revising the material.\n\n"

            "## 3. Practice Questions\n"
            "Create useful practice questions based on the supplied topics.\n\n"

            "## 4. Study Schedule\n"
            "Suggest how the student can organize their study time.\n\n"

            "## 5. Exam Tips\n"
            "Give practical exam-day and preparation advice.\n\n"

            "## 6. Common Mistakes\n"
            "List mistakes students should avoid.\n\n"

            "## 7. Final Revision Checklist\n"
            "Create a short checklist for the final revision.\n\n"

            "Focus on understanding concepts, active recall, "
            "practice, and revision."
        )

        result = generate_response(
            prompt,
            max_new_tokens=900
        )

        return result

    except Exception as e:

        return (
            "⚠️ Exam preparation failed.\n\n"
            "Error: " + str(e)
        )


print("✅ CELL 15 SUCCESSFULLY LOADED")
print("🎯 Exam Preparation is ready.")

✅ CELL 15 SUCCESSFULLY LOADED
🎯 Exam Preparation is ready.


In [20]:
# ============================================================
# CELL 16 — AI TUTOR
# ============================================================

tutor_sessions = {}


def tutor_start(topic, level):

    if not topic or not topic.strip():
        return "⚠️ Please enter a topic.", ""

    session_id = str(len(tutor_sessions) + 1)

    tutor_sessions[session_id] = {
        "topic": topic,
        "level": level,
        "stage": 1,
        "history": []
    }

    prompt = f"""
You are an interactive university tutor.

Topic:
{topic}

Student level:
{level}

Start the tutoring session.

Your first job is to ask the student what they already know about
the topic.

Do not give a long lecture yet.

Ask 1-2 diagnostic questions.
"""

    response = generate_response(
        prompt,
        max_new_tokens=400
    )

    tutor_sessions[session_id]["history"].append(
        ("Tutor", response)
    )

    return response, session_id


def tutor_continue(session_id, student_answer):

    if not session_id or session_id not in tutor_sessions:
        return "Please start a tutoring session first."

    if not student_answer or not student_answer.strip():
        return "Please enter your answer."

    session = tutor_sessions[session_id]

    topic = session["topic"]
    level = session["level"]

    session["history"].append(
        ("Student", student_answer)
    )

    previous = "\n".join(
        f"{speaker}: {message}"
        for speaker, message in session["history"][-8:]
    )

    prompt = f"""
You are an interactive university tutor.

Topic:
{topic}

Student level:
{level}

Teaching sequence:

1. Discover what the student knows.
2. Explain the concept.
3. Give an example.
4. Ask a practice question.
5. Check the student's answer.
6. Explain mistakes.
7. Continue teaching.

Recent conversation:

{previous}

Continue the tutoring session.

Do not skip the teaching process.
"""

    response = generate_response(
        prompt,
        max_new_tokens=600
    )

    session["history"].append(
        ("Tutor", response)
    )

    return response

In [21]:
# ============================================================
# CELL 17 — DASHBOARD AND SUBJECT MANAGER
# ============================================================

study_data = {
    "subjects": [],
    "study_sessions": 0,
    "notes_created": 0,
    "quizzes_generated": 0,
    "goals": []
}


def add_subject(subject):

    if not subject or not subject.strip():
        return "⚠️ Enter a subject name.", get_subjects()

    subject = subject.strip()

    if subject not in study_data["subjects"]:
        study_data["subjects"].append(subject)

    return (
        f"✅ {subject} added successfully.",
        get_subjects()
    )


def remove_subject(subject):

    if not subject:
        return "⚠️ Select a subject.", get_subjects()

    if subject in study_data["subjects"]:
        study_data["subjects"].remove(subject)

    return (
        f"Subject removed: {subject}",
        get_subjects()
    )


def get_subjects():

    if not study_data["subjects"]:
        return "No subjects added yet."

    return "\n".join(
        f"• {subject}"
        for subject in study_data["subjects"]
    )


def dashboard_data():

    return (
        str(len(study_data["subjects"])),
        str(study_data["study_sessions"]),
        str(study_data["notes_created"]),
        str(study_data["quizzes_generated"]),
        str(len(study_data["goals"]))
    )


def add_goal(goal):

    if not goal or not goal.strip():
        return get_goals()

    study_data["goals"].append(goal.strip())

    return get_goals()


def get_goals():

    if not study_data["goals"]:
        return "No study goals added yet."

    return "\n".join(
        f"🎯 {goal}"
        for goal in study_data["goals"]
    )

In [22]:
# ============================================================
# CELL 18 — CUSTOM CSS
# ============================================================

CUSTOM_CSS = """
/* ==========================================================
   AI STUDYMATE — MODERN SAAS UI
   ========================================================== */

body {
    background: #f5f7fb !important;
}

.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
}

#app-header {
    padding: 25px;
    border-radius: 18px;
    margin-bottom: 18px;
    background: linear-gradient(
        135deg,
        #111827,
        #312e81
    );
    color: white;
}

#app-title {
    font-size: 34px !important;
    font-weight: 800 !important;
    margin-bottom: 5px !important;
}

#app-subtitle {
    font-size: 16px !important;
    opacity: 0.85;
}

.sidebar {
    border-radius: 16px;
    padding: 12px;
    background: white;
}

.dashboard-card {
    background: white;
    border-radius: 16px;
    padding: 20px;
    min-height: 110px;
    border: 1px solid #e5e7eb;
}

.dashboard-number {
    font-size: 30px;
    font-weight: 800;
}

.dashboard-label {
    color: #6b7280;
    font-size: 14px;
}

.section-card {
    background: white;
    border-radius: 16px;
    padding: 20px;
    border: 1px solid #e5e7eb;
}

textarea {
    border-radius: 12px !important;
}

input {
    border-radius: 10px !important;
}

button {
    border-radius: 10px !important;
}

.primary-btn {
    font-weight: 700 !important;
}

footer {
    display: none !important;
}

@media (max-width: 768px) {

    #app-title {
        font-size: 26px !important;
    }

    .dashboard-card {
        min-height: 90px;
    }
}
"""

In [26]:
# ============================================================
# CELL 19 — COMPLETE GRADIO INTERFACE — FIXED
# Compatible with Gradio versions that don't support
# Chatbot(type="messages")
# ============================================================

with gr.Blocks(
    title="AI StudyMate"
) as app:

    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML("""
    <div id="app-header">
        <div id="app-title">🎓 AI StudyMate</div>
        <div id="app-subtitle">
            Your AI-powered university study companion
        </div>
    </div>
    """)

    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():

        # ====================================================
        # DASHBOARD
        # ====================================================

        with gr.Tab("🏠 Dashboard"):

            gr.Markdown("# Welcome back! 👋")
            gr.Markdown(
                "Organize your university studies and learn smarter."
            )

            with gr.Row():

                with gr.Column(elem_classes="dashboard-card"):
                    subjects_count = gr.Markdown("**0**")
                    gr.Markdown("### 📚 Total Subjects")

                with gr.Column(elem_classes="dashboard-card"):
                    sessions_count = gr.Markdown("**0**")
                    gr.Markdown("### ⏱️ Study Sessions")

                with gr.Column(elem_classes="dashboard-card"):
                    notes_count = gr.Markdown("**0**")
                    gr.Markdown("### 📝 Notes Created")

                with gr.Column(elem_classes="dashboard-card"):
                    quizzes_count = gr.Markdown("**0**")
                    gr.Markdown("### 🧠 Quizzes Generated")

                with gr.Column(elem_classes="dashboard-card"):
                    goals_count = gr.Markdown("**0**")
                    gr.Markdown("### 🎯 Today's Goals")

            refresh_dashboard = gr.Button(
                "🔄 Refresh Dashboard"
            )

            with gr.Row():

                with gr.Column():

                    gr.Markdown("## 🎯 Study Goals")

                    goal_input = gr.Textbox(
                        label="Add Study Goal",
                        placeholder="Example: Complete Python functions"
                    )

                    add_goal_btn = gr.Button(
                        "➕ Add Goal"
                    )

                    goals_display = gr.Markdown(
                        get_goals()
                    )

                    add_goal_btn.click(
                        add_goal,
                        inputs=goal_input,
                        outputs=goals_display
                    )

                with gr.Column():

                    gr.Markdown("## 📚 My Subjects")

                    subject_input = gr.Textbox(
                        label="Add Subject",
                        placeholder="Example: Data Structures"
                    )

                    add_subject_btn = gr.Button(
                        "➕ Add Subject"
                    )

                    subject_status = gr.Markdown()

                    subject_display = gr.Markdown(
                        get_subjects()
                    )

                    add_subject_btn.click(
                        add_subject,
                        inputs=subject_input,
                        outputs=[
                            subject_status,
                            subject_display
                        ]
                    )

            refresh_dashboard.click(
                dashboard_data,
                outputs=[
                    subjects_count,
                    sessions_count,
                    notes_count,
                    quizzes_count,
                    goals_count
                ]
            )

        # ====================================================
        # AI STUDY CHAT
        # ====================================================

        with gr.Tab("💬 AI Study Chat"):

            gr.Markdown("# 💬 AI Study Chat")
            gr.Markdown(
                "Ask AI StudyMate to explain university concepts."
            )

            with gr.Row():

                with gr.Column(scale=1):

                    chat_subject = gr.Dropdown(
                        choices=[
                            "Computer Science",
                            "Programming",
                            "Mathematics",
                            "Physics",
                            "Chemistry",
                            "Artificial Intelligence",
                            "Data Science",
                            "Software Engineering",
                            "English",
                            "Other"
                        ],
                        value="Computer Science",
                        label="Subject"
                    )

                    chat_difficulty = gr.Dropdown(
                        choices=[
                            "Beginner",
                            "Intermediate",
                            "Advanced"
                        ],
                        value="Beginner",
                        label="Difficulty"
                    )

                with gr.Column(scale=3):

                    # IMPORTANT:
                    # No type="messages"
                    chatbot = gr.Chatbot(
                        label="AI StudyMate",
                        height=500
                    )

                    chat_input = gr.Textbox(
                        label="Ask your question",
                        placeholder="Explain recursion in simple words...",
                        lines=3
                    )

                    with gr.Row():

                        send_chat = gr.Button(
                            "🚀 Ask AI",
                            variant="primary"
                        )

                        clear_chat_btn = gr.Button(
                            "🗑️ Clear Chat"
                        )

                    send_chat.click(
                        study_chat,
                        inputs=[
                            chat_input,
                            chatbot,
                            chat_subject,
                            chat_difficulty
                        ],
                        outputs=[
                            chatbot,
                            chat_input
                        ]
                    )

                    chat_input.submit(
                        study_chat,
                        inputs=[
                            chat_input,
                            chatbot,
                            chat_subject,
                            chat_difficulty
                        ],
                        outputs=[
                            chatbot,
                            chat_input
                        ]
                    )

                    clear_chat_btn.click(
                        clear_chat,
                        outputs=chatbot
                    )

        # ====================================================
        # NOTES
        # ====================================================

        with gr.Tab("📝 Notes"):

            gr.Markdown("# 📝 Notes Generator")

            with gr.Row():

                with gr.Column():

                    notes_subject = gr.Textbox(
                        label="Subject",
                        placeholder="Computer Science"
                    )

                    notes_topic = gr.Textbox(
                        label="Topic",
                        placeholder="Object-Oriented Programming"
                    )

                    notes_difficulty = gr.Dropdown(
                        [
                            "Beginner",
                            "Intermediate",
                            "Advanced"
                        ],
                        value="Intermediate",
                        label="Difficulty"
                    )

                    notes_length = gr.Dropdown(
                        [
                            "Short",
                            "Medium",
                            "Detailed"
                        ],
                        value="Medium",
                        label="Desired Length"
                    )

                    notes_btn = gr.Button(
                        "✨ Generate Notes",
                        variant="primary"
                    )

                with gr.Column():

                    notes_output = gr.Markdown(
                        "Your generated notes will appear here."
                    )

            notes_btn.click(
                generate_notes,
                inputs=[
                    notes_subject,
                    notes_topic,
                    notes_difficulty,
                    notes_length
                ],
                outputs=notes_output
            )

        # ====================================================
        # SUMMARIZER
        # ====================================================

        with gr.Tab("📄 Summarizer"):

            gr.Markdown("# 📄 Study Material Summarizer")

            summary_input = gr.Textbox(
                label="Paste Lecture Notes / Study Material",
                placeholder="Paste your study material here...",
                lines=15
            )

            summary_btn = gr.Button(
                "📌 Summarize",
                variant="primary"
            )

            summary_output = gr.Markdown()

            summary_btn.click(
                summarize_text,
                inputs=summary_input,
                outputs=summary_output
            )

        # ====================================================
        # QUIZ
        # ====================================================

        with gr.Tab("🧠 Quiz Generator"):

            gr.Markdown("# 🧠 Quiz Generator")

            with gr.Row():

                quiz_subject = gr.Textbox(
                    label="Subject",
                    placeholder="Programming"
                )

                quiz_topic = gr.Textbox(
                    label="Topic",
                    placeholder="Python Functions"
                )

            with gr.Row():

                quiz_number = gr.Slider(
                    minimum=1,
                    maximum=15,
                    value=5,
                    step=1,
                    label="Number of Questions"
                )

                quiz_difficulty = gr.Dropdown(
                    [
                        "Beginner",
                        "Intermediate",
                        "Advanced"
                    ],
                    value="Intermediate",
                    label="Difficulty"
                )

            quiz_btn = gr.Button(
                "🧠 Generate Quiz",
                variant="primary"
            )

            quiz_output = gr.Markdown()

            quiz_btn.click(
                generate_quiz,
                inputs=[
                    quiz_subject,
                    quiz_topic,
                    quiz_number,
                    quiz_difficulty
                ],
                outputs=quiz_output
            )

        # ====================================================
        # FLASHCARDS
        # ====================================================

        with gr.Tab("🎴 Flashcards"):

            gr.Markdown("# 🎴 Flashcard Generator")

            flash_subject = gr.Textbox(
                label="Subject"
            )

            flash_topic = gr.Textbox(
                label="Topic"
            )

            with gr.Row():

                flash_number = gr.Slider(
                    minimum=1,
                    maximum=20,
                    value=10,
                    step=1,
                    label="Number of Cards"
                )

                flash_difficulty = gr.Dropdown(
                    [
                        "Beginner",
                        "Intermediate",
                        "Advanced"
                    ],
                    value="Intermediate",
                    label="Difficulty"
                )

            flash_btn = gr.Button(
                "🎴 Generate Flashcards",
                variant="primary"
            )

            flash_output = gr.Markdown()

            flash_btn.click(
                generate_flashcards,
                inputs=[
                    flash_subject,
                    flash_topic,
                    flash_number,
                    flash_difficulty
                ],
                outputs=flash_output
            )

        # ====================================================
        # STUDY PLANNER
        # ====================================================

        with gr.Tab("📅 Study Planner"):

            gr.Markdown("# 📅 Study Planner")

            subjects_input = gr.Textbox(
                label="Subjects",
                placeholder="Programming, Mathematics, Physics"
            )

            exam_date_input = gr.Textbox(
                label="Exam Date",
                placeholder="September 20, 2026"
            )

            with gr.Row():

                daily_time_input = gr.Textbox(
                    label="Daily Study Time",
                    placeholder="3 hours"
                )

                study_days_input = gr.Slider(
                    minimum=1,
                    maximum=60,
                    value=14,
                    step=1,
                    label="Number of Study Days"
                )

            weak_subjects_input = gr.Textbox(
                label="Weak Subjects",
                placeholder="Mathematics, Physics"
            )

            strong_subjects_input = gr.Textbox(
                label="Strong Subjects",
                placeholder="English, Programming"
            )

            planner_btn = gr.Button(
                "📅 Create Study Plan",
                variant="primary"
            )

            planner_output = gr.Markdown()

            planner_btn.click(
                generate_study_plan,
                inputs=[
                    subjects_input,
                    exam_date_input,
                    daily_time_input,
                    weak_subjects_input,
                    strong_subjects_input,
                    study_days_input
                ],
                outputs=planner_output
            )

        # ====================================================
        # ASSIGNMENT HELPER
        # ====================================================

        with gr.Tab("📚 Assignment Helper"):

            gr.Markdown("# 📚 Assignment Helper")

            assignment_subject = gr.Textbox(
                label="Subject"
            )

            assignment_question = gr.Textbox(
                label="Assignment Question",
                lines=8,
                placeholder="Paste your assignment question..."
            )

            assignment_type = gr.Dropdown(
                [
                    "Understand the question",
                    "Create an outline",
                    "Explain the concepts",
                    "Improve organization",
                    "Check grammar",
                    "Develop my own answer"
                ],
                value="Understand the question",
                label="What do you need help with?"
            )

            assignment_btn = gr.Button(
                "📚 Get Help",
                variant="primary"
            )

            assignment_output = gr.Markdown()

            assignment_btn.click(
                assignment_helper,
                inputs=[
                    assignment_question,
                    assignment_subject,
                    assignment_type
                ],
                outputs=assignment_output
            )

        # ====================================================
        # CODE EXPLAINER
        # ====================================================

        with gr.Tab("💻 Code Explainer"):

            gr.Markdown("# 💻 Code Explainer")

            code_language = gr.Dropdown(
                [
                    "Python",
                    "C",
                    "C++",
                    "Java",
                    "JavaScript",
                    "HTML/CSS",
                    "SQL"
                ],
                value="Python",
                label="Programming Language"
            )

            code_input = gr.Code(
                label="Paste Your Code",
                language="python",
                lines=18
            )

            code_btn = gr.Button(
                "🔍 Explain Code",
                variant="primary"
            )

            code_output = gr.Markdown()

            code_btn.click(
                explain_code,
                inputs=[
                    code_input,
                    code_language
                ],
                outputs=code_output
            )

        # ====================================================
        # EXAM PREPARATION
        # ====================================================

        with gr.Tab("🎯 Exam Prep"):

            gr.Markdown("# 🎯 Exam Preparation")

            exam_subject = gr.Textbox(
                label="Subject",
                placeholder="Data Structures"
            )

            exam_topics = gr.Textbox(
                label="Topics",
                placeholder="Arrays, Linked Lists, Stacks, Queues",
                lines=6
            )

            exam_date = gr.Textbox(
                label="Exam Date",
                placeholder="September 25, 2026"
            )

            exam_difficulty = gr.Dropdown(
                [
                    "Beginner",
                    "Intermediate",
                    "Advanced"
                ],
                value="Intermediate",
                label="Difficulty"
            )

            exam_btn = gr.Button(
                "🎯 Create Exam Plan",
                variant="primary"
            )

            exam_output = gr.Markdown()

            exam_btn.click(
                exam_preparation,
                inputs=[
                    exam_subject,
                    exam_topics,
                    exam_date,
                    exam_difficulty
                ],
                outputs=exam_output
            )

        # ====================================================
        # AI TUTOR
        # ====================================================

        with gr.Tab("👨‍🏫 AI Tutor"):

            gr.Markdown("# 👨‍🏫 Interactive AI Tutor")

            gr.Markdown(
                "Diagnose → Explain → Example → Practice → "
                "Check → Correct → Continue"
            )

            tutor_topic = gr.Textbox(
                label="Topic",
                placeholder="Example: Recursion"
            )

            tutor_level = gr.Dropdown(
                [
                    "Beginner",
                    "Intermediate",
                    "Advanced"
                ],
                value="Beginner",
                label="Your Level"
            )

            tutor_start_btn = gr.Button(
                "🚀 Start Tutor",
                variant="primary"
            )

            tutor_session = gr.Textbox(
                label="Session ID",
                interactive=False
            )

            tutor_output = gr.Markdown()

            tutor_start_btn.click(
                tutor_start,
                inputs=[
                    tutor_topic,
                    tutor_level
                ],
                outputs=[
                    tutor_output,
                    tutor_session
                ]
            )

            tutor_answer = gr.Textbox(
                label="Your Answer",
                placeholder="Type your answer here...",
                lines=5
            )

            tutor_continue_btn = gr.Button(
                "➡️ Continue Lesson"
            )

            tutor_continue_btn.click(
                tutor_continue,
                inputs=[
                    tutor_session,
                    tutor_answer
                ],
                outputs=tutor_output
            )

        # ====================================================
        # SUBJECT MANAGER
        # ====================================================

        with gr.Tab("📚 Subjects"):

            gr.Markdown("# 📚 Subject Manager")

            subject_manager_input = gr.Textbox(
                label="Subject Name",
                placeholder="Artificial Intelligence"
            )

            add_manager_btn = gr.Button(
                "➕ Add Subject"
            )

            manager_status = gr.Markdown()

            manager_subjects = gr.Markdown(
                get_subjects()
            )

            add_manager_btn.click(
                add_subject,
                inputs=subject_manager_input,
                outputs=[
                    manager_status,
                    manager_subjects
                ]
            )

    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown(
        """
        ---
        ### 🎓 AI StudyMate

        **Your AI-powered university study companion**

        🤖 Local Hugging Face AI • 🔒 No API Key • 🚫 No Paid API
        """
    )


print("✅ CELL 19 SUCCESSFULLY LOADED")
print("🎓 AI StudyMate interface has been created.")

✅ CELL 19 SUCCESSFULLY LOADED
🎓 AI StudyMate interface has been created.


In [27]:
# ============================================================
# CELL 20 — LAUNCH AI STUDYMATE
# ============================================================

print("=" * 60)
print("🚀 STARTING AI STUDYMATE")
print("=" * 60)

if not model_loaded:

    print("""
⚠️ The AI model was not loaded successfully.

The interface can still be created, but AI generation will not work.

Please:
1. Check the error from Cell 5.
2. Restart the runtime if necessary.
3. Run Cells 1–6 again.
4. Then run Cell 20.
""")

else:

    print("✅ AI model is loaded.")
    print("✅ Starting Gradio...")
    print("🌐 A public Gradio URL will appear below.")

app.launch(
    share=True
)

🚀 STARTING AI STUDYMATE
✅ AI model is loaded.
✅ Starting Gradio...
🌐 A public Gradio URL will appear below.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://374be75be884e2fcc0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
